<h1># Decision Tree Model Training</h1>

## 1. Import the required libraries

In [ ]:
import json
from pathlib import Path
import joblib

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

## 2. Load the preprocessed training and testing data

In [ ]:
# Find the main project folder from the current notebook working directory.
current_folder = Path.cwd()
project_folder = None

for folder in [current_folder] + list(current_folder.parents):
    if (folder / "data" / "processed").exists():
        project_folder = folder
        break

if project_folder is None:
    raise FileNotFoundError("Could not find data/processed. Open Jupyter from the main project folder.")

data_folder = project_folder / "data" / "processed"
model_folder = project_folder / "models" / "decision_tree"

X_train = joblib.load(data_folder / "X_train_processed.pkl")
X_test = joblib.load(data_folder / "X_test_processed.pkl")
y_train = joblib.load(data_folder / "y_train.pkl")
y_test = joblib.load(data_folder / "y_test.pkl")

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Number of features:", X_train.shape[1])


## 3. Create the model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=20, min_samples_leaf=10, class_weight="balanced", random_state=42)

print(model)

## 4. Train the model

In [ ]:
model.fit(X_train, y_train)
print("Training completed.")

## 5. Evaluate the model

In [ ]:
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)
roc_auc = roc_auc_score(y_test, probabilities)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

## 6. Save the trained model and metrics

In [ ]:
model_folder.mkdir(parents=True, exist_ok=True)

metrics = {
    "model": model.__class__.__name__,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "roc_auc": float(roc_auc),
    "confusion_matrix": confusion_matrix(y_test, predictions).tolist(),
    "train_rows": int(X_train.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "processed_features": int(X_train.shape[1]),
    "positive_class_definition": "1 = readmitted within 30 days; 0 = not readmitted within 30 days"
}

joblib.dump(model, model_folder / "model.joblib")

with open(model_folder / "metrics.json", "w") as file:
    json.dump(metrics, file, indent=4)

print("Model saved:", model_folder / "model.joblib")
print("Metrics saved:", model_folder / "metrics.json")

## 7. Conclusion

Compare this model's scores with the other two models. Do not choose a model using accuracy alone; consider recall and F1 score.